# Week 3: Improved Methods to Beat 87%

## V1 Baseline: 87% accuracy (6 errors)

**Alternative approaches tested:**
1. MLP Probe (non-linear classifier)
2. 3-Token Lookahead (more context)
3. Additional Layers (more features)
4. Ensemble of Probes

**Persistent errors from V1:**
- FN: ORM library, web server, model trained, CI/CD pipeline
- FP: book I read, mental state

In [1]:
# Cell 1: Install
!pip install -q transformers torch accelerate scipy scikit-learn pandas matplotlib seaborn

In [2]:
# Cell 2: Imports
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import entropy as scipy_entropy
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
print("Imports complete")

Imports complete


In [3]:
# Cell 3: Load Model
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    output_hidden_states=True
)
model.eval()

print(f"Model loaded on {model.device}")

Loading codellama/CodeLlama-7b-Instruct-hf...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded on cuda:0


In [4]:
# Cell 4: Load training data

LABELED_DATA_FILE = '/content/training_data_combined_FIXED.csv'  # Colab
# LABELED_DATA_FILE = 'E:/SummerProjects/reposynth/research/training_data_combined_FIXED.csv'  # Local

print(f"Loading {LABELED_DATA_FILE}...")
df = pd.read_csv(LABELED_DATA_FILE)

valid_labels = df['label'].isin(['code', 'language'])
df = df[valid_labels]
df['label_binary'] = df['label'].map({'language': 0, 'code': 1})

code_count = (df['label_binary']==1).sum()
lang_count = (df['label_binary']==0).sum()

print(f"\nLoaded {len(df)} training examples")
print(f"   LANGUAGE: {lang_count} ({lang_count/len(df)*100:.1f}%)")
print(f"   CODE: {code_count} ({code_count/len(df)*100:.1f}%)")

Loading /content/training_data_combined_FIXED.csv...

Loaded 1780 training examples
   LANGUAGE: 1353 (76.0%)
   CODE: 427 (24.0%)


In [5]:
# Cell 5: Hidden state extraction functions

def get_multi_layer_state(text: str, layers: List[int]) -> np.ndarray:
    """Extract and concatenate hidden states from multiple layers."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    states = [
        outputs.hidden_states[layer_idx + 1][:, -1, :].cpu().numpy()[0]
        for layer_idx in layers
    ]
    return np.concatenate(states).astype(np.float32)

print("Hidden state extraction ready")

Hidden state extraction ready


## Method 1: MLP Probe (Non-linear Classifier)

LogisticRegression is linear. An MLP can capture more complex patterns.

In [6]:
# Cell 6: Extract hidden states for training (V1 layers)

V1_LAYERS = [8, 16, 31]

print(f"Extracting hidden states from layers {V1_LAYERS}...")

X_train = []
y_train = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting"):
    h = get_multi_layer_state(row['full_text'], V1_LAYERS)
    X_train.append(h)
    y_train.append(row['label_binary'])

X_train = np.array(X_train)
y_train = np.array(y_train)

print(f"\nHidden states extracted: {X_train.shape}")

Extracting hidden states from layers [8, 16, 31]...


Extracting:   0%|          | 0/1780 [00:00<?, ?it/s]


Hidden states extracted: (1780, 12288)


In [7]:
# Cell 7: Compare LogisticRegression vs MLP

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# V1: LogisticRegression
lr_probe = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_scores = cross_val_score(lr_probe, X_train_scaled, y_train, cv=cv)
print(f"LogisticRegression CV: {lr_scores.mean():.1%} (+/- {lr_scores.std()*2:.1%})")

# MLP: Various architectures
mlp_configs = [
    {'hidden': (128,), 'name': 'MLP (128)'},
    {'hidden': (256,), 'name': 'MLP (256)'},
    {'hidden': (128, 64), 'name': 'MLP (128, 64)'},
    {'hidden': (256, 128), 'name': 'MLP (256, 128)'},
    {'hidden': (512, 256, 128), 'name': 'MLP (512, 256, 128)'},
]

best_mlp = None
best_mlp_score = 0

for config in mlp_configs:
    mlp = MLPClassifier(
        hidden_layer_sizes=config['hidden'],
        max_iter=500,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20
    )
    scores = cross_val_score(mlp, X_train_scaled, y_train, cv=cv)
    print(f"{config['name']} CV: {scores.mean():.1%} (+/- {scores.std()*2:.1%})")

    if scores.mean() > best_mlp_score:
        best_mlp_score = scores.mean()
        best_mlp = config

print(f"\nBest MLP: {best_mlp['name']} with {best_mlp_score:.1%}")

LogisticRegression CV: 95.4% (+/- 3.5%)
MLP (128) CV: 95.2% (+/- 2.2%)
MLP (256) CV: 94.7% (+/- 3.2%)
MLP (128, 64) CV: 95.6% (+/- 2.4%)
MLP (256, 128) CV: 95.6% (+/- 3.7%)
MLP (512, 256, 128) CV: 95.5% (+/- 3.1%)

Best MLP: MLP (256, 128) with 95.6%


In [8]:
# Cell 8: Train best probes

# Train LogisticRegression
lr_probe = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_probe.fit(X_train_scaled, y_train)

# Train best MLP
mlp_probe = MLPClassifier(
    hidden_layer_sizes=best_mlp['hidden'],
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20
)
mlp_probe.fit(X_train_scaled, y_train)

print("Both probes trained")

Both probes trained


In [9]:
# Cell 9: Helper functions

def softmax(logits: np.ndarray) -> np.ndarray:
    logits_stable = logits - np.max(logits)
    exp_logits = np.exp(logits_stable)
    return exp_logits / np.sum(exp_logits)

def entropy_from_probs(probs: np.ndarray) -> float:
    if len(probs) == 0 or np.sum(probs) == 0:
        return 0.0
    norm_probs = probs / np.sum(probs)
    return float(scipy_entropy(norm_probs, base=2))

def get_next_tokens(text: str, n: int = 1) -> List[str]:
    """Get the next n most likely tokens."""
    tokens = []
    current = text
    for _ in range(n):
        inputs = tokenizer(current, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits[0, -1, :].cpu().numpy()
        next_token_id = np.argmax(logits)
        next_token = tokenizer.decode([next_token_id])
        tokens.append(next_token)
        current = current + next_token
    return tokens

print("Helper functions ready")

Helper functions ready


In [10]:
# Cell 10: Test cases (same as V1)

CODE_TEST_CASES = [
    {'prompt': 'In our React app, authentication is done using', 'category': 'auth_method'},
    {'prompt': 'In the backend, passwords are hashed with', 'category': 'auth_hash'},
    {'prompt': 'For our API, JWT tokens are signed using', 'category': 'auth_signing'},
    {'prompt': 'In production, the OAuth provider we use is', 'category': 'auth_provider'},
    {'prompt': 'On the server, session data is stored in', 'category': 'session_store'},
    {'prompt': 'For data persistence, the database we use is', 'category': 'database_type'},
    {'prompt': 'In the application, we query the database using', 'category': 'database_query'},
    {'prompt': 'For database access, the ORM library is', 'category': 'database_orm'},
    {'prompt': 'To improve performance, caching is implemented with', 'category': 'database_cache'},
    {'prompt': 'For the REST API, the framework we use is', 'category': 'web_framework'},
    {'prompt': 'In production, the web server runs on', 'category': 'web_server'},
    {'prompt': 'In the client code, HTTP requests are made using', 'category': 'http_client'},
    {'prompt': 'For data fetching, our GraphQL server uses', 'category': 'graphql_server'},
    {'prompt': 'For the UI, the frontend framework is', 'category': 'frontend_framework'},
    {'prompt': 'In the application, state management is handled by', 'category': 'frontend_state'},
    {'prompt': 'For the interface, components are built with', 'category': 'frontend_components'},
    {'prompt': 'In the SPA, routing is done using', 'category': 'frontend_routing'},
    {'prompt': 'For training, the model is trained with', 'category': 'ml_framework'},
    {'prompt': 'In our neural network, deep learning is implemented using', 'category': 'ml_deep_learning'},
    {'prompt': 'For gradient descent, the optimizer we use is', 'category': 'ml_optimizer'},
    {'prompt': 'For hosting, we deploy to', 'category': 'cloud_platform'},
    {'prompt': 'In Kubernetes, containers are orchestrated with', 'category': 'cloud_containers'},
    {'prompt': 'For automation, the CI/CD pipeline uses', 'category': 'cloud_cicd'},
    {'prompt': 'In the test suite, unit tests are written with', 'category': 'test_unit'},
    {'prompt': 'For building assets, the bundler we use is', 'category': 'build_bundler'},
    {'prompt': 'For dependencies, package management is done with', 'category': 'build_package_manager'},
]

LANGUAGE_TEST_CASES = [
    {'prompt': 'The weather today is', 'category': 'description_weather'},
    {'prompt': 'The meeting yesterday was', 'category': 'description_meeting'},
    {'prompt': 'My favorite color has always been', 'category': 'description_color'},
    {'prompt': 'The book I read last week was', 'category': 'description_book'},
    {'prompt': 'The movie we watched seemed', 'category': 'description_movie'},
    {'prompt': 'The main idea of the story is to', 'category': 'explanation_idea'},
    {'prompt': 'The cooking process works by', 'category': 'explanation_process'},
    {'prompt': 'This teaching approach helps to', 'category': 'explanation_approach'},
    {'prompt': 'The benefit of exercise is', 'category': 'explanation_benefit'},
    {'prompt': 'When installing furniture in my home, you should', 'category': 'instruction_furniture'},
    {'prompt': 'To debug a relationship problem, first', 'category': 'instruction_debug'},
    {'prompt': 'Before deploying troops, the general needs to', 'category': 'instruction_deploy'},
    {'prompt': 'The configuration of the room requires', 'category': 'instruction_config'},
    {'prompt': 'To optimize your morning routine, try', 'category': 'instruction_optimize'},
    {'prompt': 'The framework of the argument is', 'category': 'nontechnical_framework'},
    {'prompt': 'My mental state is managed by', 'category': 'nontechnical_state'},
    {'prompt': 'The library in town has', 'category': 'nontechnical_library'},
    {'prompt': 'Running the business takes', 'category': 'nontechnical_running'},
    {'prompt': 'The function of the heart is', 'category': 'nontechnical_function'},
    {'prompt': 'Implementing the new policy will', 'category': 'nontechnical_implement'},
]

ALL_TEST_CASES = [
    {**case, 'expected_stop': True} for case in CODE_TEST_CASES
] + [
    {**case, 'expected_stop': False} for case in LANGUAGE_TEST_CASES
]

print(f"Test cases: {len(ALL_TEST_CASES)} (CODE: {len(CODE_TEST_CASES)}, LANGUAGE: {len(LANGUAGE_TEST_CASES)})")

Test cases: 46 (CODE: 26, LANGUAGE: 20)


In [11]:
# Cell 11: Lookahead analysis function (supports 2 or 3 tokens)

def analyze_lookahead(prompt: str, probe, lookahead: int = 2, top_k: int = 10) -> Dict:
    """
    N-token lookahead analysis.
    Returns code_votes (sum of probs where probe predicts CODE).
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :].cpu().numpy()

    probs = softmax(logits)
    top_indices = np.argsort(probs)[-top_k:][::-1]

    candidates = []
    code_votes = 0
    lang_votes = 0

    for idx in top_indices:
        token = tokenizer.decode([idx])
        prob = probs[idx]

        # Build N-token lookahead sequence
        sequence = prompt + token
        for _ in range(lookahead - 1):
            next_tokens = get_next_tokens(sequence, 1)
            sequence += next_tokens[0]

        # Classify the full sequence
        h = get_multi_layer_state(sequence, V1_LAYERS).reshape(1, -1)
        h_scaled = scaler.transform(h)
        token_type = probe.predict(h_scaled)[0]
        type_prob = probe.predict_proba(h_scaled)[0, 1]

        candidates.append({
            'token': token,
            'prob': prob,
            'sequence': sequence,
            'type': 'CODE' if token_type == 1 else 'LANGUAGE',
            'type_prob': type_prob
        })

        if token_type == 1:
            code_votes += prob
        else:
            lang_votes += prob

    return {
        'candidates': candidates,
        'code_votes': code_votes,
        'lang_votes': lang_votes,
    }

print("Lookahead analysis ready")

Lookahead analysis ready


## Run All Methods and Compare

In [12]:
# Cell 12: Collect results for all methods

print("Collecting results for all method combinations...")
print("Methods: LR-2tok, MLP-2tok, LR-3tok, MLP-3tok")
print()

results = []

for test_case in tqdm(ALL_TEST_CASES, desc="Testing"):
    prompt = test_case['prompt']

    # LR + 2-token
    lr_2tok = analyze_lookahead(prompt, lr_probe, lookahead=2)

    # MLP + 2-token
    mlp_2tok = analyze_lookahead(prompt, mlp_probe, lookahead=2)

    # LR + 3-token
    lr_3tok = analyze_lookahead(prompt, lr_probe, lookahead=3)

    # MLP + 3-token
    mlp_3tok = analyze_lookahead(prompt, mlp_probe, lookahead=3)

    results.append({
        'prompt': prompt,
        'category': test_case['category'],
        'expected_stop': test_case['expected_stop'],
        'lr_2tok_votes': lr_2tok['code_votes'],
        'mlp_2tok_votes': mlp_2tok['code_votes'],
        'lr_3tok_votes': lr_3tok['code_votes'],
        'mlp_3tok_votes': mlp_3tok['code_votes'],
    })

df_results = pd.DataFrame(results)
print(f"\nCollected {len(df_results)} test cases")

Methods: LR-2tok, MLP-2tok, LR-3tok, MLP-3tok



Testing:   0%|          | 0/46 [00:00<?, ?it/s]


Collected 46 test cases


In [13]:
# Cell 13: Find optimal threshold for each method

def find_best_threshold(df, vote_column, threshold_range=np.arange(0.05, 0.60, 0.01)):
    """Find optimal threshold for a given vote column."""
    best_acc = 0
    best_thresh = 0
    best_metrics = {}

    for thresh in threshold_range:
        predicted = df[vote_column] > thresh
        correct = (predicted == df['expected_stop']).sum()
        accuracy = correct / len(df)

        tp = ((df['expected_stop'] == True) & (predicted == True)).sum()
        fp = ((df['expected_stop'] == False) & (predicted == True)).sum()
        fn = ((df['expected_stop'] == True) & (predicted == False)).sum()
        tn = ((df['expected_stop'] == False) & (predicted == False)).sum()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        if accuracy > best_acc:
            best_acc = accuracy
            best_thresh = thresh
            best_metrics = {
                'accuracy': accuracy,
                'precision': precision,
                'recall': recall,
                'f1': f1,
                'errors': len(df) - correct,
                'fn': fn,
                'fp': fp
            }

    return best_thresh, best_metrics

# Find best threshold for each method
methods = [
    ('LR + 2-token', 'lr_2tok_votes'),
    ('MLP + 2-token', 'mlp_2tok_votes'),
    ('LR + 3-token', 'lr_3tok_votes'),
    ('MLP + 3-token', 'mlp_3tok_votes'),
]

print("\n" + "="*80)
print("RESULTS COMPARISON")
print("="*80)
print(f"\nV1 Baseline: 87.0% accuracy (threshold=0.30, 6 errors: 4 FN, 2 FP)")
print("\n" + "-"*80)

method_results = {}

for name, col in methods:
    thresh, metrics = find_best_threshold(df_results, col)
    method_results[name] = {'threshold': thresh, **metrics}

    improvement = "+" if metrics['accuracy'] > 0.87 else ""
    print(f"\n{name}:")
    print(f"  Threshold: {thresh:.2f}")
    print(f"  Accuracy:  {metrics['accuracy']:.1%} {improvement}")
    print(f"  F1 Score:  {metrics['f1']:.1%}")
    print(f"  Errors:    {metrics['errors']} (FN: {metrics['fn']}, FP: {metrics['fp']})")


RESULTS COMPARISON

V1 Baseline: 87.0% accuracy (threshold=0.30, 6 errors: 4 FN, 2 FP)

--------------------------------------------------------------------------------

LR + 2-token:
  Threshold: 0.05
  Accuracy:  91.3% +
  F1 Score:  92.3%
  Errors:    4 (FN: 2, FP: 2)

MLP + 2-token:
  Threshold: 0.05
  Accuracy:  89.1% +
  F1 Score:  90.2%
  Errors:    5 (FN: 3, FP: 2)

LR + 3-token:
  Threshold: 0.05
  Accuracy:  84.8% 
  F1 Score:  86.8%
  Errors:    7 (FN: 3, FP: 4)

MLP + 3-token:
  Threshold: 0.05
  Accuracy:  87.0% 
  F1 Score:  88.0%
  Errors:    6 (FN: 4, FP: 2)


In [14]:
# Cell 14: Summary table

print("\n" + "="*80)
print("SUMMARY TABLE")
print("="*80)

print(f"\n| Method         | Threshold | Accuracy | F1     | Errors | FN | FP | vs V1   |")
print(f"|----------------|-----------|----------|--------|--------|----|----|---------|")
print(f"| V1 Baseline    | 0.30      |   87.0%  |   --   |      6 |  4 |  2 |    --   |")

for name, data in method_results.items():
    diff = (data['accuracy'] - 0.87) * 100
    diff_str = f"+{diff:.1f}%" if diff > 0 else f"{diff:.1f}%"
    print(f"| {name:<14} | {data['threshold']:.2f}      | {data['accuracy']*100:>6.1f}%  | {data['f1']*100:>5.1f}% | {data['errors']:>6} | {data['fn']:>2} | {data['fp']:>2} | {diff_str:>7} |")


SUMMARY TABLE

| Method         | Threshold | Accuracy | F1     | Errors | FN | FP | vs V1   |
|----------------|-----------|----------|--------|--------|----|----|---------|
| V1 Baseline    | 0.30      |   87.0%  |   --   |      6 |  4 |  2 |    --   |
| LR + 2-token   | 0.05      |   91.3%  |  92.3% |      4 |  2 |  2 |   +4.3% |
| MLP + 2-token  | 0.05      |   89.1%  |  90.2% |      5 |  3 |  2 |   +2.1% |
| LR + 3-token   | 0.05      |   84.8%  |  86.8% |      7 |  3 |  4 |   -2.2% |
| MLP + 3-token  | 0.05      |   87.0%  |  88.0% |      6 |  4 |  2 |   -0.0% |


In [15]:
# Cell 15: Find the best overall method

best_method = max(method_results.items(), key=lambda x: x[1]['accuracy'])

print("\n" + "="*80)
print("BEST METHOD")
print("="*80)

print(f"\n{best_method[0]}")
print(f"  Threshold: {best_method[1]['threshold']:.2f}")
print(f"  Accuracy:  {best_method[1]['accuracy']:.1%}")
print(f"  F1 Score:  {best_method[1]['f1']:.1%}")
print(f"  Errors:    {best_method[1]['errors']}")

if best_method[1]['accuracy'] > 0.87:
    improvement = (best_method[1]['accuracy'] - 0.87) * 100
    print(f"\n*** IMPROVEMENT over V1: +{improvement:.1f}% ***")
else:
    print(f"\nNo improvement over V1 baseline (87%)")


BEST METHOD

LR + 2-token
  Threshold: 0.05
  Accuracy:  91.3%
  F1 Score:  92.3%
  Errors:    4

*** IMPROVEMENT over V1: +4.3% ***


In [16]:
# Cell 16: Analyze errors with best method

best_name, best_data = best_method
vote_col = [col for name, col in methods if name == best_name][0]

df_results['best_predicted'] = df_results[vote_col] > best_data['threshold']
df_results['best_correct'] = df_results['best_predicted'] == df_results['expected_stop']

errors = df_results[~df_results['best_correct']]

print(f"\n{'='*80}")
print(f"ERROR ANALYSIS ({best_name}, threshold={best_data['threshold']:.2f})")
print(f"{'='*80}")

if len(errors) > 0:
    fn_errors = errors[errors['expected_stop'] == True]
    fp_errors = errors[errors['expected_stop'] == False]

    if len(fn_errors) > 0:
        print(f"\nFALSE NEGATIVES ({len(fn_errors)}) - Should STOP but didn't:")
        for _, row in fn_errors.iterrows():
            print(f"  votes={row[vote_col]:.3f} | '{row['prompt']}'")

    if len(fp_errors) > 0:
        print(f"\nFALSE POSITIVES ({len(fp_errors)}) - Should CONTINUE but stopped:")
        for _, row in fp_errors.iterrows():
            print(f"  votes={row[vote_col]:.3f} | '{row['prompt']}'")
else:
    print("\nPERFECT! No errors!")


ERROR ANALYSIS (LR + 2-token, threshold=0.05)

FALSE NEGATIVES (2) - Should STOP but didn't:
  votes=0.000 | 'For database access, the ORM library is'
  votes=0.015 | 'For training, the model is trained with'

FALSE POSITIVES (2) - Should CONTINUE but stopped:
  votes=0.179 | 'The book I read last week was'
  votes=0.354 | 'My mental state is managed by'


## Method 2: Ensemble (Combine Multiple Probes)

In [17]:
# Cell 17: Ensemble method - combine LR and MLP votes

print("\n" + "="*80)
print("ENSEMBLE METHOD")
print("="*80)

# Try various ensemble combinations
ensemble_methods = [
    ('LR+MLP 2tok (avg)', ['lr_2tok_votes', 'mlp_2tok_votes']),
    ('LR+MLP 3tok (avg)', ['lr_3tok_votes', 'mlp_3tok_votes']),
    ('LR 2+3tok (avg)', ['lr_2tok_votes', 'lr_3tok_votes']),
    ('MLP 2+3tok (avg)', ['mlp_2tok_votes', 'mlp_3tok_votes']),
    ('All 4 (avg)', ['lr_2tok_votes', 'mlp_2tok_votes', 'lr_3tok_votes', 'mlp_3tok_votes']),
]

print(f"\n| Ensemble Method    | Threshold | Accuracy | F1     | Errors |")
print(f"|--------------------|-----------|----------|--------|--------|")

for name, cols in ensemble_methods:
    # Average the votes
    df_results[f'ensemble_{name}'] = df_results[cols].mean(axis=1)
    thresh, metrics = find_best_threshold(df_results, f'ensemble_{name}')

    print(f"| {name:<18} | {thresh:.2f}      | {metrics['accuracy']*100:>6.1f}%  | {metrics['f1']*100:>5.1f}% | {metrics['errors']:>6} |")


ENSEMBLE METHOD

| Ensemble Method    | Threshold | Accuracy | F1     | Errors |
|--------------------|-----------|----------|--------|--------|
| LR+MLP 2tok (avg)  | 0.05      |   91.3%  |  92.3% |      4 |
| LR+MLP 3tok (avg)  | 0.06      |   87.0%  |  88.5% |      6 |
| LR 2+3tok (avg)    | 0.05      |   87.0%  |  88.9% |      6 |
| MLP 2+3tok (avg)   | 0.05      |   89.1%  |  90.6% |      5 |
| All 4 (avg)        | 0.05      |   89.1%  |  90.6% |      5 |


## Method 3: Different Layer Combinations

In [18]:
# Cell 18: Test different layer combinations

LAYER_CONFIGS = [
    [8, 16, 31],           # V1 (baseline)
    [4, 8, 16, 24, 31],    # More layers
    [16, 24, 31],          # Later layers only
    [24, 28, 31],          # Very late layers
    [8, 16, 24, 31],       # Add layer 24
]

print("\n" + "="*80)
print("LAYER COMBINATION COMPARISON")
print("="*80)
print("\nTraining probes with different layer combinations...")

layer_results = {}

for layers in LAYER_CONFIGS:
    layer_name = str(layers)
    print(f"\nTesting layers {layer_name}...")

    # Extract hidden states for these layers
    X_layers = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting", leave=False):
        h = get_multi_layer_state(row['full_text'], layers)
        X_layers.append(h)
    X_layers = np.array(X_layers)

    # Scale
    layer_scaler = StandardScaler()
    X_layers_scaled = layer_scaler.fit_transform(X_layers)

    # Train probe
    layer_probe = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    cv_scores = cross_val_score(layer_probe, X_layers_scaled, y_train, cv=cv)
    layer_probe.fit(X_layers_scaled, y_train)

    print(f"  CV accuracy: {cv_scores.mean():.1%}")

    # Test on test cases
    layer_votes = []
    for test_case in tqdm(ALL_TEST_CASES, desc="Testing", leave=False):
        prompt = test_case['prompt']

        # 2-token lookahead with this probe
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits[0, -1, :].cpu().numpy()

        probs = softmax(logits)
        top_indices = np.argsort(probs)[-10:][::-1]

        code_votes = 0
        for idx in top_indices:
            token = tokenizer.decode([idx])
            prob = probs[idx]

            # 2-token sequence
            sequence = prompt + token
            next_tokens = get_next_tokens(sequence, 1)
            sequence += next_tokens[0]

            # Classify
            h = get_multi_layer_state(sequence, layers).reshape(1, -1)
            h_scaled = layer_scaler.transform(h)
            token_type = layer_probe.predict(h_scaled)[0]

            if token_type == 1:
                code_votes += prob

        layer_votes.append({
            'prompt': prompt,
            'expected_stop': test_case['expected_stop'],
            'code_votes': code_votes
        })

    df_layer = pd.DataFrame(layer_votes)
    thresh, metrics = find_best_threshold(df_layer, 'code_votes')

    layer_results[layer_name] = {
        'cv_accuracy': cv_scores.mean(),
        'threshold': thresh,
        **metrics
    }

    print(f"  Test accuracy: {metrics['accuracy']:.1%} (threshold={thresh:.2f})")


LAYER COMBINATION COMPARISON

Training probes with different layer combinations...

Testing layers [8, 16, 31]...


Extracting:   0%|          | 0/1780 [00:00<?, ?it/s]

  CV accuracy: 95.4%


Testing:   0%|          | 0/46 [00:00<?, ?it/s]

  Test accuracy: 91.3% (threshold=0.05)

Testing layers [4, 8, 16, 24, 31]...


...

  Test accuracy: 91.3% (threshold=0.06)

Testing layers [8, 16, 24, 31]...


Extracting:   0%|          | 0/1780 [00:00<?, ?it/s]

  CV accuracy: 95.1%


Testing:   0%|          | 0/46 [00:00<?, ?it/s]

  Test accuracy: 91.3% (threshold=0.06)


In [19]:
# Cell 19: Layer results summary

print("\n" + "="*80)
print("LAYER COMBINATION RESULTS")
print("="*80)

print(f"\n| Layers              | CV Acc | Threshold | Test Acc | Errors | vs V1   |")
print(f"|---------------------|--------|-----------|----------|--------|---------|")

for layer_name, data in layer_results.items():
    diff = (data['accuracy'] - 0.87) * 100
    diff_str = f"+{diff:.1f}%" if diff > 0 else f"{diff:.1f}%"
    marker = " <--" if layer_name == "[8, 16, 31]" else ""
    print(f"| {layer_name:<19} | {data['cv_accuracy']*100:>5.1f}% | {data['threshold']:.2f}      | {data['accuracy']*100:>6.1f}%  | {data['errors']:>6} | {diff_str:>7} |{marker}")


LAYER COMBINATION RESULTS

| Layers              | CV Acc | Threshold | Test Acc | Errors | vs V1   |
|---------------------|--------|-----------|----------|--------|---------|
| [8, 16, 31]         |  95.4% | 0.05      |   91.3%  |      4 |   +4.3% | <--
| [4, 8, 16, 24, 31]  |  95.0% | 0.08      |   91.3%  |      4 |   +4.3% |
| [16, 24, 31]        |  95.4% | 0.06      |   93.5%  |      3 |   +6.5% |
| [24, 28, 31]        |  95.1% | 0.06      |   91.3%  |      4 |   +4.3% |
| [8, 16, 24, 31]     |  95.1% | 0.06      |   91.3%  |      4 |   +4.3% |


In [20]:
# Cell 20: Final summary

print("\n" + "="*80)
print("FINAL SUMMARY: METHODS TO IMPROVE BEYOND 87%")
print("="*80)

all_results = []

# Add method results
for name, data in method_results.items():
    all_results.append({
        'method': name,
        'accuracy': data['accuracy'],
        'errors': data['errors']
    })

# Add layer results
for layers, data in layer_results.items():
    all_results.append({
        'method': f'Layers {layers}',
        'accuracy': data['accuracy'],
        'errors': data['errors']
    })

# Sort by accuracy
all_results.sort(key=lambda x: x['accuracy'], reverse=True)

print(f"\nRanked by accuracy:")
print(f"\n| Rank | Method                        | Accuracy | Errors | vs 87% |")
print(f"|------|-------------------------------|----------|--------|--------|")

for i, res in enumerate(all_results[:10], 1):
    diff = (res['accuracy'] - 0.87) * 100
    diff_str = f"+{diff:.1f}%" if diff > 0 else f"{diff:.1f}%"
    print(f"| {i:>4} | {res['method']:<29} | {res['accuracy']*100:>6.1f}%  | {res['errors']:>6} | {diff_str:>6} |")

best_overall = all_results[0]
print(f"\n{'='*80}")
if best_overall['accuracy'] > 0.87:
    improvement = (best_overall['accuracy'] - 0.87) * 100
    print(f"BEST: {best_overall['method']} with {best_overall['accuracy']:.1%} (+{improvement:.1f}% vs V1)")
else:
    print(f"No method beats 87%. Best is {best_overall['method']} at {best_overall['accuracy']:.1%}")
    print(f"\nThe 6 errors appear to be fundamental limitations of the approach.")
    print(f"Consider: more training data, different model, or hybrid approaches.")


FINAL SUMMARY: METHODS TO IMPROVE BEYOND 87%

Ranked by accuracy:

| Rank | Method                        | Accuracy | Errors | vs 87% |
|------|-------------------------------|----------|--------|--------|
|    1 | Layers [16, 24, 31]           |   93.5%  |      3 |  +6.5% |
|    2 | LR + 2-token                  |   91.3%  |      4 |  +4.3% |
|    3 | Layers [8, 16, 31]            |   91.3%  |      4 |  +4.3% |
|    4 | Layers [4, 8, 16, 24, 31]     |   91.3%  |      4 |  +4.3% |
|    5 | Layers [24, 28, 31]           |   91.3%  |      4 |  +4.3% |
|    6 | Layers [8, 16, 24, 31]        |   91.3%  |      4 |  +4.3% |
|    7 | MLP + 2-token                 |   89.1%  |      5 |  +2.1% |
|    8 | MLP + 3-token                 |   87.0%  |      6 |  -0.0% |
|    9 | LR + 3-token                  |   84.8%  |      7 |  -2.2% |

BEST: Layers [16, 24, 31] with 93.5% (+6.5% vs V1)


In [21]:
# Cell 21: Save all results

import json

# Compile all results
final_report = {
    'v1_baseline': {'accuracy': 0.87, 'errors': 6, 'threshold': 0.30},
    'probe_methods': method_results,
    'layer_combinations': layer_results,
    'best_method': {
        'name': best_overall['method'],
        'accuracy': best_overall['accuracy'],
        'errors': best_overall['errors'],
        'improvement_vs_v1': (best_overall['accuracy'] - 0.87) * 100
    }
}

# Convert numpy types to Python types for JSON
def convert_types(obj):
    if isinstance(obj, dict):
        return {k: convert_types(v) for k, v in obj.items()}
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

final_report = convert_types(final_report)

with open('improved_methods_results.json', 'w') as f:
    json.dump(final_report, f, indent=2)

df_results.to_csv('improved_methods_test_results.csv', index=False)

print("Results saved to:")
print("  - improved_methods_results.json")
print("  - improved_methods_test_results.csv")

Results saved to:
  - improved_methods_results.json
  - improved_methods_test_results.csv
